### SQL analytical queries

In [1]:
import sqlite3
import pandas as pd

In [3]:
conn = sqlite3.connect('db/cafe_sales.db')
df = pd.read_sql_query("SELECT * FROM sales", conn)

In [4]:
df.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08 00:00:00
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16 00:00:00
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19 00:00:00
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27 00:00:00
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11 00:00:00


In [13]:
#Calcular o faturamento total por dia e o crescimento acumulado ao longo do tempo.
query = '''
SELECT transaction_date, SUM(total_spent) AS total_spent, SUM(SUM(total_spent)) OVER (ORDER BY transaction_date) AS total_spent_per_day
FROM sales
GROUP BY transaction_date
ORDER BY transaction_date
'''

df = pd.read_sql_query(query, conn)
df.head(10)

,transaction_date,total_spent,total_spent_per_day
0,2023-01-01 00:00:00,156.0,156.0
1,2023-01-02 00:00:00,189.5,345.5
2,2023-01-03 00:00:00,194.0,539.5
3,2023-01-04 00:00:00,241.5,781.0
4,2023-01-05 00:00:00,340.5,1121.5
5,2023-01-06 00:00:00,225.0,1346.5
6,2023-01-07 00:00:00,126.5,1473.0
7,2023-01-08 00:00:00,281.5,1754.5
8,2023-01-09 00:00:00,176.0,1930.5
9,2023-01-10 00:00:00,238.5,2169.0


In [37]:
#Para cada dia, rankear os itens mais vendidos.
query = '''
SELECT transaction_date, item, SUM(quantity) AS total_quantity,DENSE_RANK() OVER (PARTITION BY transaction_date ORDER BY SUM(quantity) DESC) AS rank_per_day
FROM sales
GROUP BY transaction_date, item
ORDER BY transaction_date, rank_per_day;
'''

df = pd.read_sql_query(query, conn)
df.head(10)

,transaction_date,item,total_quantity,rank_per_day
0,2023-01-01 00:00:00,Coffee,14.0,1
1,2023-01-01 00:00:00,Cake,10.0,2
2,2023-01-01 00:00:00,Tea,8.0,3
3,2023-01-01 00:00:00,Sandwich,6.0,4
4,2023-01-01 00:00:00,Juice,6.0,4
5,2023-01-01 00:00:00,Unknown,4.0,5
6,2023-01-01 00:00:00,Salad,4.0,5
7,2023-01-01 00:00:00,Cookie,4.0,5
8,2023-01-01 00:00:00,Smoothie,2.0,6
9,2023-01-02 00:00:00,Sandwich,16.0,1
